In [1]:
# the same standard stuff as we did in DWSIM_Model_Eval to jumpstart the dlls and interpretor 

In [2]:
import pandas as pd
pd.set_option("display.precision", 12)

In [3]:
import clr 
import sys

In [4]:
sys.path.append(r"C:\Users\Admin\AppData\Local\DWSIM")
clr.AddReference("DWSIM.Automation")
clr.AddReference("DWSIM.Interfaces")

from DWSIM.Automation import Automation3

automation = Automation3()

In [5]:
flowsheet = automation.LoadFlowsheet(
    r"C:\Users\Admin\AppData\Local\DWSIM\bdc.dwxmz"
)
print("Flowsheet Loaded !")

Flowsheet Loaded !


In [6]:
units = flowsheet.FlowsheetOptions.SelectedUnitSystem

In [7]:
#accesssing feed stream, column , up stream and bottom stream by initiating them as follow
feed = flowsheet.GetFlowsheetSimulationObject("Feed") #feed stream was implicitly names as "1" i named it to feed in dwsim 
col = flowsheet.GetFlowsheetSimulationObject("DCOL-1") #column initiation 
dis= flowsheet.GetFlowsheetSimulationObject("disStream") #upstream initiation
btm = flowsheet.GetFlowsheetSimulationObject("bottomStream") #Bottom stream initiation 

In [8]:
col_obj = flowsheet.GetFlowsheetSimulationObject("DCOL-1").GetAsObject()

In [9]:
#making a funtion to plug in values from a dataframe and returns required results so that we can iterate through inputs and store results

In [10]:
import time
def run_case_mode(row):
    # 1. Reset calculation flags
    for obj in flowsheet.SimulationObjects.Values:
        obj.Calculated = False

    # 2. Update inputs
    feed.SetPropertyValue("PROP_MS_0", row["Feed temperature"], units)
    feed.SetPropertyValue("PROP_MS_1", row["Feed pressure"], units)
    feed.SetPropertyValue("PROP_MS_102/Benzene", row["Feed composition"], units)
    feed.SetPropertyValue("PROP_MS_102/Toluene", 1.0 - row["Feed composition"], units)
    feed.SetPropertyValue("PROP_MS_3", row["Molar flow"], units)

    col.SetPropertyValue("PROP_DC_07", row["Number of stages"], units)
    col.SetPropertyValue("Condenser_Specification_Value", row["Reflux ratio"], units)
    col.SetPropertyValue("Reboiler_Specification_Value", row["Bottoms withdrawal rate"], units)
    col.SetPropertyValue("Stage_Pressure_1", row["Column pressure"], units)
    col.SetPropertyValue("Stream 'Feed' Stage Index", row["Feed Stage"], units)

    # 3. Synchronous solve
    errors = automation.CalculateFlowsheet2(flowsheet)

    # 4. CHECK IF COLUMN CONVERGED
    # In DWSIM, check if column calculation succeeded
    is_converged = col.Converged if hasattr(col, "Converged") else True

    if not is_converged:
        # Return None or NaN for unconverged runs
        return {
            "Distillate purity": None,
            "Bottom purity": None,
            "Condensor Duty": None,
            "Reboiler Duty": None,
            "Converged": False
        }

    # 5. Read outputs for converged runs
    valueDdp = dis.GetPropertyValue("PROP_MS_102/Benzene", units)
    valueBbp = btm.GetPropertyValue("PROP_MS_102/Toluene", units)
    valueDcd = col.GetPropertyValue("PROP_DC_5", units)
    valueDrd = col.GetPropertyValue("PROP_DC_6", units)

    return {
        "Distillate purity": valueDdp,
        "Bottom purity": valueBbp,
        "Condensor Duty": valueDcd,
        "Reboiler Duty": valueDrd,
        "Converged": True
    }

In [11]:
# reading input values from csv file we generated in sampling file and turning it into a data frame 
input1 = pd.read_csv("4kinput")

In [12]:
input1.head()

,Feed temperature,Feed pressure,Feed composition,Number of stages,Reflux ratio,Bottoms withdrawal rate,Molar flow,Column pressure,Feed Stage
0,329.475389786072,1.066954205635,0.253453046545,31,2.219618568923,2.717731766386,3.460038172202,1.536538341602,20
1,322.117829495617,2.813161005799,0.238419614433,16,2.573842851783,4.893620575950,6.432262393922,2.448703468268,10
2,330.801948178929,1.904004633174,0.215137230218,18,1.994736585730,1.326680183004,1.536966193739,1.981899872181,11
3,368.338860143819,1.852736588577,0.431655593880,39,2.997926763979,2.379202345887,3.920070297690,1.818977760369,20
4,329.719518097057,1.676681861084,0.303382678299,39,2.340097219750,2.140200263936,3.118924220997,1.537434541306,13


In [13]:
#looping through input and calling func. "run_case" for each input while storing the results in results1
results = []
for idx, row in input1.iterrows():
    out = run_case_mode(row)
    
    # Safely keep ONLY runs where output is valid and not None
    if out is not None and out.get("Distillate purity") is not None:
        combined = {**row.to_dict(), **out}
        results.append(combined)
        
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} / {len(input1)} rows... Retained {len(results)} valid converged runs.")
# Convert to DataFrame & Save
df_clean = pd.DataFrame(results)
# Remove any duplicate output rows just to be 100% clean
df_clean.drop_duplicates(subset=["Distillate purity", "Condensor Duty"], inplace=True)
df_clean.to_csv("dwsim_surrogate_clean.csv", index=False)
print(f"\nSUCCESS! Saved {len(df_clean)} 100% unique, converged rows to dwsim_surrogate_clean.csv")

Processed 100 / 4000 rows... Retained 100 valid converged runs.
Processed 200 / 4000 rows... Retained 200 valid converged runs.
Processed 300 / 4000 rows... Retained 300 valid converged runs.
Processed 400 / 4000 rows... Retained 400 valid converged runs.
Processed 500 / 4000 rows... Retained 500 valid converged runs.
Processed 600 / 4000 rows... Retained 600 valid converged runs.
Processed 700 / 4000 rows... Retained 700 valid converged runs.
Processed 800 / 4000 rows... Retained 800 valid converged runs.
Processed 900 / 4000 rows... Retained 900 valid converged runs.
Processed 1000 / 4000 rows... Retained 1000 valid converged runs.
Processed 1100 / 4000 rows... Retained 1100 valid converged runs.
Processed 1200 / 4000 rows... Retained 1200 valid converged runs.
Processed 1300 / 4000 rows... Retained 1300 valid converged runs.
Processed 1400 / 4000 rows... Retained 1400 valid converged runs.
Processed 1500 / 4000 rows... Retained 1500 valid converged runs.
Processed 1600 / 4000 rows..

In [15]:
resf = pd.DataFrame(results)
# SUCCESS! Saved 1219 100% unique, converged rows to dwsim_surrogate_clean.csv 
    # just saving the console log of above code run for reference 

In [16]:
resf.head()

,Feed temperature,Feed pressure,Feed composition,Number of stages,Reflux ratio,Bottoms withdrawal rate,Molar flow,Column pressure,Feed Stage,Distillate purity,Bottom purity,Condensor Duty,Reboiler Duty,Converged
0,329.475389786072,1.066954205635,0.253453046545,31.0,2.219618568923,2.717731766386,3.460038172202,1.536538341602,20.0,0.999999883892,0.78448503200,-21.541342653278,21.851808182051,True
1,322.117829495617,2.813161005799,0.238419614433,16.0,2.573842851783,4.893620575950,6.432262393922,2.448703468268,10.0,0.999999883892,0.78448503200,-21.541342653278,21.851808182051,True
2,330.801948178929,1.904004633174,0.215137230218,18.0,1.994736585730,1.326680183004,1.536966193739,1.981899872181,11.0,0.999999933569,0.90926914747,-4.055541644511,9.357068967872,True
3,368.338860143819,1.852736588577,0.431655593880,39.0,2.997926763979,2.379202345887,3.920070297690,1.818977760369,20.0,0.999999933569,0.90926914747,-4.055541644511,9.357068967872,True
4,329.719518097057,1.676681861084,0.303382678299,39.0,2.340097219750,2.140200263936,3.118924220997,1.537434541306,13.0,0.999999933569,0.90926914747,-4.055541644511,9.357068967872,True


In [19]:
#stripping any erroric spaces in lale bname due to typo
resf.columns = resf.columns.str.strip()

In [20]:
# in previous 2 attempts this >=0.999 value concerned me since 55% of dataset is pinned right against the asymptotic ceiling
# (≥0.999)To a standard Random Forest, 0.9990,0.9999 and 0.99999 look virtually identical (a tiny 0.0009 difference). But in physical reality, reaching 
# 0.9999 requires drastically higher reflux and reboiler duties than 0.9990
# so i tried to change data set such that high values of purity dont come in data set but even the clipped ranges gave 
# massive spiked at value >0.999 thats when it occured to me that i have to find a way to turn the flat small 0.0009 difference 
# into a massive linearized target signal so that its easier for machine to discint 

In [21]:
# thats when the sigmoid function came into this since it is to like out distillate purity strictly confined to value range
# [0,1] hence we are, instead of drawing output on Xd(distillate purity)  , going to use the logit transform and draw
# output on a variable say Xknot = log e(Xd/(1-Xd)) and then return Xd as Xd = 1/(1 + e^(Xknot))